# M3-C Research Runner — Colab Run All (bounded smoke on Core 699e3ad + Drive archive)

This notebook runs one reviewed **research configuration**, `configs/research/sma_5_20_tsm_conservative_synthetic_v1.json`. It is the M3-C Phase C bounded smoke test of the Agent ↔ Core M3-A.4 integration:

* fixed SMA 5/20 on the committed **native-1m TEST_ONLY synthetic fixture**;
* Core `CONSERVATIVE_LABEL_AGNOSTIC` temporal model (START/END envelopes, RTH only, fills strictly later);
* **TEST_ONLY** componentized costs (BASE / LOW / STRESS_2X) over 2 windows.

It runs the configuration **once**, then a **resume** pass that must skip every run. It then verifies the Google Drive archive and checks numeric reproducibility against the committed reference metrics.

Flow: Spec → Governance → Data (Core `prepare_conservative_bars`) → Signal → Core `ExecutionEngine.run` → Evaluation → Registry → Report → Archive.

* **Nothing local needed.** Open the notebook from GitHub in Colab and press Run All. Python ≥ 3.11 is checked.
* **Pinned code.** The agent is cloned at the exact, already-committed integration commit `AGENT_COMMIT`. This notebook pins that commit and is committed *after* it, so there is no self-referential SHA. Core is installed at the exact `CORE_SHA` (`BENCHMARK_PASS`).
* **No strategy, backtest or cost logic here.** Cells call `python -m strategy_agent.research` and check files. TEST_ONLY costs are not approved rates. Synthetic results say nothing about TSM.
* **Credentials.** None are needed: both repositories are public. An optional `GITHUB_TOKEN` is read only from **Colab Secrets** and is never printed or persisted.
* **Isolation.** The run uses an isolated governance and registry under `/content/m3c_env`. The repository's `registry/` and `governance/` are fingerprinted before and after the run.
* **Real TSM stays blocked.** The market gate is **NO_GO**. The readiness cell must exit 3.
* **Reproducibility (N1).** The numeric check uses a tolerance fixed in code. Each archive must match its own SHAs exactly. Bitwise equality across environments is reported, never forced.
* **Archive location.** `MyDrive/Quant_Research/strategy-research-agent/YYYYMMDD_HHMMSS_<commit7>/` (Asia/Taipei).


In [ ]:
# --- configuration (pinned; change only together with a reviewed commit) ---
AGENT_REPO   = "https://github.com/hh4832/strategy-research-agent.git"
AGENT_COMMIT = "PENDING_INTEGRATION_COMMIT"                # exact 40-hex integration commit, never a branch
CORE_SHA     = "699e3ad59951bbebb769bb69170be8d9a1a96c01"   # Core M3-A.4 (BENCHMARK_PASS on this SHA)
WORKDIR      = "/content/strategy-research-agent"
RUN_ROOT     = "/content/m3c_env"                       # isolated synthetic governance + registry (not on Drive)
CONFIG       = "configs/research/sma_5_20_tsm_conservative_synthetic_v1.json"   # reviewed config (DRIVE output)
REFERENCE    = "docs/evidence/m3c_reference_metrics.json"  # committed reference metrics (N1 numeric check)
DRIVE_ROOT   = "/content/drive/MyDrive/Quant_Research/strategy-research-agent"
import hashlib, json, os, pathlib, re, subprocess, sys
assert sys.version_info >= (3, 11), f"Python >= 3.11 required, found {sys.version}"
assert re.fullmatch(r"[0-9a-f]{40}", AGENT_COMMIT), "AGENT_COMMIT must be a full commit SHA"
assert re.fullmatch(r"[0-9a-f]{40}", CORE_SHA)


In [ ]:
# --- Google Drive (required: the archive target). Run All stops here if Drive cannot be mounted ---
from google.colab import drive
drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), "Google Drive is not mounted"


In [ ]:
# --- optional Colab Secret (not needed for public repos). Never printed or written to disk ---
GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = None
print("GITHUB_TOKEN from Colab Secrets:", "present" if GITHUB_TOKEN else "not set (public clone)")


In [ ]:
# --- clone the agent at AGENT_COMMIT, install it and Core at CORE_SHA ---
MASK = [t for t in (GITHUB_TOKEN,) if t]
def sh(cmd, secret_env=None, check=True):
    print("$", cmd)
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0", **(secret_env or {}))   # never wait for a password prompt
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, env=env)
    out = (r.stdout[-6000:] + r.stderr[-3000:])
    for t in MASK:
        out = out.replace(t, "***")
    print(out)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")
    return r
sh(f"rm -rf {WORKDIR}")
# the repository is public: clone WITHOUT credentials first; a Colab Secret is used only if that fails
r = sh(f"git clone -q {AGENT_REPO} {WORKDIR}", check=False)
if r.returncode != 0:
    assert GITHUB_TOKEN, "public clone failed and no GITHUB_TOKEN in Colab Secrets"
    import base64
    basic = base64.b64encode(f"x-access-token:{GITHUB_TOKEN}".encode()).decode()
    MASK.append(basic)
    sh(f"rm -rf {WORKDIR}")
    # header passed via env for this one command only; not stored in .git/config
    sh(f'git -c http.extraHeader="Authorization: Basic $GH_BASIC" clone -q {AGENT_REPO} {WORKDIR}',
       secret_env={"GH_BASIC": basic})
sh(f"cd {WORKDIR} && git checkout -q {AGENT_COMMIT}")
head = sh(f"cd {WORKDIR} && git rev-parse HEAD").stdout.strip()
assert head == AGENT_COMMIT, (head, AGENT_COMMIT)
assert sh(f"cd {WORKDIR} && git status --porcelain").stdout.strip() == "", "checkout is not clean"
assert "extraheader" not in open(f"{WORKDIR}/.git/config").read().lower(), "credential persisted in .git/config"
sh(f"cd {WORKDIR} && pip install -q -e '.[test,research]' "
   f"'market-research-core @ git+https://github.com/hh4832/market-research-core@{CORE_SHA}'")


In [ ]:
# --- identity: agent commit, Core commit, BENCHMARK_PASS derived from the committed evidence record ---
r = sh(f"cd {WORKDIR} && python -c \"import json; from strategy_agent.adapters import mrc_adapter as A; "
       f"from strategy_agent.research.pipeline import agent_identity; "
       f"print(json.dumps({{'core': A.core_identity(), 'engine': A.engine_info().__dict__, 'agent': agent_identity()}}))\"")
ident = json.loads(r.stdout.strip().splitlines()[-1])
assert ident["core"]["git_sha"] == CORE_SHA and ident["engine"]["benchmark_status"] == "BENCHMARK_PASS"
assert ident["agent"]["agent_sha"] == AGENT_COMMIT and ident["agent"]["agent_tree_dirty"] is False


In [ ]:
# --- tests: M3-C Core M3-A.4 integration (14 items), M3-B contract, M1 Core compatibility on the real Core ---
sh(f"cd {WORKDIR} && python -m pytest -q tests/test_m3c_core_integration.py tests/test_m3b2_contract.py tests/test_m1_core_compat.py")


In [ ]:
# --- the market gate stays closed: the real TSM run must refuse (exit 3) ---
r = sh(f"cd {WORKDIR} && python -m strategy_agent.research readiness --market IBKR_TSM", check=False)
assert r.returncode == 3, "IBKR_TSM gate is expected to be closed (formal market NO_GO)"
r = sh(f"cd {WORKDIR} && python -m strategy_agent.research real --market IBKR_TSM", check=False)
assert r.returncode == 3 and "NOT STARTED" in r.stdout


In [ ]:
# --- formal registry / holdout fingerprint BEFORE the runs ---
def formal_digest():
    root = pathlib.Path(WORKDIR)
    files = sorted(list((root / "registry").glob("*")) + list((root / "governance").rglob("*")))
    return {str(p.relative_to(root)): hashlib.sha256(p.read_bytes()).hexdigest() for p in files if p.is_file()}
FORMAL_BEFORE = formal_digest()
assert not pathlib.Path(RUN_ROOT).resolve().is_relative_to(pathlib.Path(WORKDIR).resolve())
print(len(FORMAL_BEFORE), "formal files fingerprinted")


In [ ]:
# --- run 1 (the only execution): 2 windows x BASE/LOW/STRESS_2X on Core 699e3ad, archived to Drive ---
from datetime import datetime
from zoneinfo import ZoneInfo
BATCH = f"colab_m3c_{datetime.now(ZoneInfo('Asia/Taipei')):%Y%m%d_%H%M%S}"
r = sh(f"cd {WORKDIR} && python -m strategy_agent.research run --config {CONFIG} --run-root {RUN_ROOT} "
       f"--batch-id {BATCH} --summary-out /content/m3c_run1.json", check=False)
assert r.returncode == 0, "run 1 failed or Drive archive unavailable (see output)"
S1 = json.load(open('/content/m3c_run1.json'))
assert S1["config"]["sha256"] and S1["config"]["kind"] == "SYNTHETIC"
RUNS1 = S1['runs']
assert len(RUNS1) == 6 and all(o['status'] == 'COMPLETED' and o['archive_status'] == 'ARCHIVED' for o in RUNS1)
DIRS1 = [pathlib.Path(o['archive_path']) for o in RUNS1]
assert all(d.is_relative_to(pathlib.Path(DRIVE_ROOT)) for d in DIRS1), DIRS1
SNAP1 = {str(d): {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in d.iterdir()} for d in DIRS1}
print(len(DIRS1), "runs archived:", [d.name for d in DIRS1])


In [ ]:
# --- resume pass on the same batch: every identity is COMPLETED+ARCHIVED, so nothing may run again ---
r = sh(f"cd {WORKDIR} && python -m strategy_agent.research run --config {CONFIG} --run-root {RUN_ROOT} "
       f"--batch-id {BATCH} --resume --summary-out /content/m3c_run2.json", check=False)
assert r.returncode == 0, "resume pass failed (see output)"
S2 = json.load(open('/content/m3c_run2.json'))
RUNS2 = S2['runs']
print([o['status'] for o in RUNS2])
# --- N1 numeric reproducibility vs the committed reference (tolerance fixed in code; bitwise only reported) ---
r = sh(f"cd {WORKDIR} && python -m strategy_agent.research repro --summary /content/m3c_run1.json "
       f"--reference {REFERENCE} --out /content/m3c_repro.json", check=False)
REPRO = json.load(open('/content/m3c_repro.json'))


In [ ]:
# --- acceptance checks (Drive archive) ---
REQUIRED = ["run_info.txt", "experiment_record.json", "stages.json", "metrics.json", "research_report.md",
            "trades.csv", "fills.csv", "rejections.csv", "equity_curve.csv", "snapshot_refs.json",
            "final_status.json", "manifest.json", "research_config.json", "cost_scenario.json"]
checks = {}
def manifest_ok(d):
    man = json.loads((d / "manifest.json").read_text())["files"]
    return set(man) == {p.name for p in d.iterdir()} - {"manifest.json"} and \
        all(hashlib.sha256((d / f).read_bytes()).hexdigest() == v["sha256"] for f, v in man.items())
def stage(d, name):
    return {s["name"]: s for s in json.loads((d / "stages.json").read_text())}[name]
checks["run1_required_files"] = all((d / f).is_file() for d in DIRS1 for f in REQUIRED)
checks["run1_manifest_every_file_sha256"] = all(manifest_ok(d) for d in DIRS1)
checks["run1_engineering_pass"] = all(json.loads((d / "final_status.json").read_text())["status"] == "PASS" for d in DIRS1)
checks["run1_on_drive"] = all(d.is_relative_to(pathlib.Path(DRIVE_ROOT)) for d in DIRS1)
checks["core_sha_in_provenance"] = all(CORE_SHA in (d / "snapshot_refs.json").read_text() for d in DIRS1)
checks["core_contract_checks_pass"] = all(set(stage(d, "core_execution")["outputs"]["contract_checks"].values()) == {"PASS"}
                                          for d in DIRS1)
checks["core_m3a4_checks_pass"] = all(set(stage(d, "core_execution")["outputs"]["core_m3a4_contract_checks"].values())
                                      <= {"PASS", "NOT_APPLICABLE_OPEN_POSITION", "NOT_APPLICABLE_NO_ROUND_TRIPS"}
                                      for d in DIRS1)
checks["conservative_policy_on_every_fill"] = all(
    "CONSERVATIVE_LABEL_AGNOSTIC" in (d / "fills.csv").read_text() and ",UNKNOWN," in (d / "fills.csv").read_text()
    for d in DIRS1)
checks["resume_skips_every_run"] = [o["status"] for o in RUNS2] == ["SKIPPED_RESUME"] * len(RUNS1)
checks["run1_not_overwritten"] = all({p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in d.iterdir()} == SNAP1[str(d)]
                                     for d in DIRS1)
checks["archive_integrity_self_sha"] = all(v["archive_integrity"] == "PASS" for v in REPRO["runs"].values())
checks["numeric_reproducibility_vs_reference"] = REPRO["ok"] is True
checks["cost_scenarios_one_family_trial"] = S1["cost_sensitivity"]["counts_as_new_family_or_trial"] is False
checks["formal_registry_holdout_untouched"] = formal_digest() == FORMAL_BEFORE
checks["git_tree_clean"] = sh(f"cd {WORKDIR} && git status --porcelain").stdout.strip() == ""
checks["run_info_mentions_commit"] = all(AGENT_COMMIT in (d / "run_info.txt").read_text() for d in DIRS1)
BITWISE = {k: v["bitwise_cross_environment"] for k, v in REPRO["runs"].items()}   # reported, never asserted
print(json.dumps(checks, indent=1)); print("bitwise vs reference:", BITWISE)
assert all(checks.values()), [k for k, v in checks.items() if not v]


In [ ]:
# --- write the acceptance record next to the archives (Drive) ---
acc_dir = pathlib.Path(DRIVE_ROOT) / "_acceptance"
acc_dir.mkdir(parents=True, exist_ok=True)
acc = {"acceptance": "M3-C Phase C bounded Colab smoke + Drive archive (Core M3-A.4)", "result": "PASS",
       "agent_commit": AGENT_COMMIT, "core_sha": CORE_SHA, "config": S1["config"], "python": sys.version,
       "batch_id": BATCH, "run1": [str(d) for d in DIRS1], "resume": [o["status"] for o in RUNS2],
       "checks": checks, "bitwise_vs_reference": BITWISE, "repro": REPRO,
       "at": datetime.now(ZoneInfo("Asia/Taipei")).isoformat()}
p = acc_dir / f"m3c_colab_{datetime.now(ZoneInfo('Asia/Taipei')):%Y%m%d_%H%M%S}_{AGENT_COMMIT[:7]}.json"
p.write_text(json.dumps(acc, indent=1, default=str))
print("M3-C Colab smoke / Drive archive acceptance: PASS ->", p)


In [ ]:
# --- show the cost-sensitivity table and one archived report ---
from IPython.display import Markdown, display
display(Markdown(pathlib.Path(S1["summary_path"]).with_name(f"cost_sensitivity_{S1['batch_id']}.md").read_text()))
print((DIRS1[0] / "run_info.txt").read_text())
display(Markdown((DIRS1[0] / "research_report.md").read_text()))
